In [33]:

from platypus import NSGAII, Real, Integer, Problem
import puggles as pg 
from numba import njit, prange, set_num_threads, get_num_threads
import puggles as pg

set_num_threads(8)  # Choose a value no larger than your logical CPU count.



In [49]:
def objective_function(x):
    return [(x[0]-3)**2 + (x[1]-5)**2]  # must return a LIST, one entry per objective


def solve_puggles(execution_mode='sequential'):
    p = pg.Problem(solution_length=2, number_of_objectives=1,
                   solution_data_types=[pg.Real(-100.0, 100.0)] * 2,
                   objective_function=objective_function, direction=[-1])
    ga = pg.NSGAII(p, population_size=100, execution_mode=execution_mode); ga.run(1_000_000)
    return [(s.objectives[0], s.variables) for s in ga.get_archive()]


# solve_puggles()


In [43]:
def solve_platypus(): 
    p = Problem(2, 1)
    p.types[:] = Real(-100,100)
    p.function  = objective_function
    a = NSGAII(p, population_size=100)
    a.run(100_000)

# solve_platypus()

In [44]:
%%timeit
solve_platypus()

8.67 s ± 79.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [50]:
%%timeit
solve_puggles()

641 ms ± 11.2 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [54]:
import numpy as np
from numba import njit, prange, set_num_threads, get_num_threads
import puggles as pg

set_num_threads(8)  # Choose a value no larger than your logical CPU count.

@njit(parallel=True, nogil=True)
def objective_batch_native(xs):
    out = np.empty((xs.shape[0], 1), dtype=np.float64)

    for i in prange(xs.shape[0]):
        dx = xs[i, 0] - 3.0
        dy = xs[i, 1] - 5.0
        out[i, 0] = dx * dx + dy * dy

    return out
    
def objective_batch(population):
    # This Python wrapper runs briefly; Numba releases the GIL in the native call.
    xs = np.ascontiguousarray(population, dtype=np.float64)
    return objective_batch_native(xs).tolist()

# Compile before timing, so JIT compilation is excluded.
objective_batch_native(np.zeros((1, 2)))

problem = pg.Problem(
    solution_length=2,
    number_of_objectives=1,
    solution_data_types=[pg.Real(-100.0, 100.0)] * 2,
    objective_function=None,
    batch_objective_function=objective_batch,
    direction=[-1],
)
ga = pg.NSGAII(
    problem,
    population_size=100,
    execution_mode="multithreaded",  # Correct: Numba owns the parallel region.
)



In [55]:
%%timeit
ga.run(1_000_000)


1.09 s ± 13.2 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
